In [7]:
import pandas as pd

df = pd.read_csv('../data/NationalNames.csv')
df

,Id,Name,Year,Gender,Count
0,1,Mary,1880,F,7065
1,2,Anna,1880,F,2604
2,3,Emma,1880,F,2003
3,4,Elizabeth,1880,F,1939
4,5,Minnie,1880,F,1746
...,...,...,...,...,...
1825428,1825429,Zykeem,2014,M,5
1825429,1825430,Zymeer,2014,M,5
1825430,1825431,Zymiere,2014,M,5
1825431,1825432,Zyran,2014,M,5


In [20]:

# 1. Group the baby names dataset by Year and Sex. Find the most popular name each year.

# 1. Sort by Count descending so the most popular name is on top
df_sorted = df.sort_values(by=['Year', 'Gender', 'Count'], ascending=[True, True, False])

# 2. Group by Year and Gender, then take the first row of each group
most_popular = df_sorted.groupby(['Year', 'Gender']).first().reset_index()

# 3. Display the clean result
print(most_popular[['Year', 'Gender', 'Name', 'Count']])

     Year Gender    Name  Count
0    1880      F    Mary   7065
1    1880      M    John   9655
2    1881      F    Mary   6919
3    1881      M    John   8769
4    1882      F    Mary   8148
..    ...    ...     ...    ...
265  2012      M   Jacob  19030
266  2013      F  Sophia  21147
267  2013      M    Noah  18179
268  2014      F    Emma  20799
269  2014      M    Noah  19144

[270 rows x 4 columns]


In [21]:
# 2. Calculate the year-over-year growth in total births.
# 1. Group by Year and sum the Count column
annual_births = df.groupby('Year')['Count'].sum()

# 2. Calculate the year-over-year percentage growth
yoy_growth = annual_births.pct_change() * 100

# 3. Combine into a clean DataFrame for viewing
result = annual_births.to_frame(name='Total Births')
result['YoY Growth (%)'] = yoy_growth
print(result)


      Total Births  YoY Growth (%)
Year                              
1880        201484             NaN
1881        192699       -4.360148
1882        221538       14.965828
1883        216950       -2.070977
1884        243467       12.222632
...            ...             ...
2010       3686589       -3.295684
2011       3646730       -1.081189
2012       3643336       -0.093070
2013       3626802       -0.453815
2014       3670151        1.195240

[135 rows x 2 columns]


In [22]:
# 3. Use transform to create a column showing percentage of total for each name within its year.

# 1. Calculate the total births for each year and broadcast it back to the rows
year_totals = df.groupby('Year')['Count'].transform('sum')

# 2. Divide individual counts by the yearly totals to get the percentage
df['Percentage_of_Year'] = (df['Count'] / year_totals) * 100

# 3. View the updated dataset
print(df[['Year', 'Name', 'Gender', 'Count', 'Percentage_of_Year']])


         Year       Name Gender  Count  Percentage_of_Year
0        1880       Mary      F   7065            3.506482
1        1880       Anna      F   2604            1.292410
2        1880       Emma      F   2003            0.994124
3        1880  Elizabeth      F   1939            0.962359
4        1880     Minnie      F   1746            0.866570
...       ...        ...    ...    ...                 ...
1825428  2014     Zykeem      M      5            0.000136
1825429  2014     Zymeer      M      5            0.000136
1825430  2014    Zymiere      M      5            0.000136
1825431  2014      Zyran      M      5            0.000136
1825432  2014      Zyrin      M      5            0.000136

[1825433 rows x 5 columns]


In [23]:
# 4. Create a cross-tabulation (pd.crosstab) of Year vs Sex.

# Cross-tabulation summing the actual birth counts
ct_births = pd.crosstab(
    index=df['Year'], 
    columns=df['Gender'], 
    values=df['Count'], 
    aggfunc='sum',
    margins=True  # Adds a row/column for total sums
)

print(ct_births)

Gender          F          M        All
Year                                   
1880        90993     110491     201484
1881        91954     100745     192699
1882       107850     113688     221538
1883       112321     104629     216950
1884       129022     114445     243467
...           ...        ...        ...
2011      1753500    1893230    3646730
2012      1753922    1889414    3643336
2013      1745339    1881463    3626802
2014      1768775    1901376    3670151
All     167070477  170064949  337135426

[136 rows x 3 columns]


In [ ]:
# 5. Challenge: Implement a custom aggregation function that returns the range (max-min) of a group. 

# 1. Define the custom range function
def get_range(series):
    return series.max() - series.min()

# 2. Apply it to a group (for example, finding the range of baby counts per year)
count_range_per_year = df.groupby('Year')['Count'].agg(get_range).reset_index()

# Rename the column for clarity
count_range_per_year.columns = ['Year', 'Count_Range']

print(count_range_per_year)

# or

# # Apply the range calculation using a lambda function
# count_range_per_year = df.groupby('Year')['Count'].agg(lambda x: x.max() - x.min()).reset_index()

     Year  Count_Range
0    1880         9650
1    1881         8764
2    1882         9552
3    1883         8889
4    1884         9383
..    ...          ...
130  2010        22878
131  2011        21811
132  2012        22262
133  2013        21142
134  2014        20794

[135 rows x 2 columns]
